In [ ]:
!pip install -q wandb

In [ ]:
# !pip install jupytext


In [1]:
# !jupytext --set-formats ipynb,py hpml_hw_2.ipynb

In [34]:
import wandb
import os
from google.colab import userdata
import numpy as np
import time
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
wandb.init(project="hpml-hw2")
wandb.run.name = "nw0_bs32_lr1e-4_epoch5_adamw_nocompile"
wandb.config.update({
    "model_name": "distilbert-base-uncased",
    "max_len": 256,
    "batch_size": 32,
    "lr": 1e-4,
    "optimizer": "AdamW",
    "num_workers": 2, # Default value, can be changed for comparison
    "epochs": 5,
    "compile_mode": False,
    "profile_run": False # New flag for profiling
})

In [35]:
import torch
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast
from datasets import load_dataset

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
  print("Device:", torch.cuda.get_device_name(0))

GPU available: True
Device: Tesla T4


In [36]:
dataset = load_dataset("imdb")

In [37]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [38]:
tokenizer = DistilBertTokenizerFast.from_pretrained(wandb.config.model_name)

# Preprocessing function
def tokenize(batch):
    return tokenizer(batch['text'], padding="max_length", truncation=True, max_length=wandb.config.max_len)

# Apply to train and test
tokenized_train = dataset["train"].map(tokenize, batched=True)
tokenized_test = dataset["test"].map(tokenize, batched=True)

In [39]:
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "label"])
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [40]:
from torch.utils.data import DataLoader

train_loader = DataLoader(tokenized_train, batch_size=wandb.config.batch_size, shuffle=True, num_workers=wandb.config.num_workers)
test_loader = DataLoader(tokenized_test, batch_size=wandb.config.batch_size, num_workers=wandb.config.num_workers)

In [41]:
batch = next(iter(train_loader))
print(batch.keys())
print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["label"])

dict_keys(['label', 'input_ids', 'attention_mask'])
torch.Size([32, 256])
torch.Size([32, 256])
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1,
        0, 1, 0, 1, 0, 1, 0, 1])


In [42]:
model = DistilBertForSequenceClassification.from_pretrained(
    wandb.config.model_name, num_labels=2
)
if torch.cuda.is_available():
  model.cuda()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [43]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [44]:
def accuracy_from_logits(logits, labels):
    preds = torch.argmax(logits, dim=-1)
    correct = (preds == labels).sum().item()
    total = labels.numel()
    return correct, total

In [45]:
from torch.optim import SGD, Adam, AdamW
if wandb.config == "adamw":
    optimizer = torch.optim.AdamW(model.parameters(), lr=wandb.config.lr)
elif wandb.config == "adam":
    optimizer = torch.optim.Adam(model.parameters(), lr=wandb.config.lr)
elif wandb.config == "sgd":
    optimizer = torch.optim.SGD(model.parameters(), lr=wandb.config.lr)
else:
    raise ValueError(f"Unknown optimizer: {wandb.config}")


ValueError: Unknown optimizer: {'model_name': 'distilbert-base-uncased', 'max_len': 256, 'batch_size': 32, 'lr': 0.0001, 'optimizer': 'AdamW', 'num_workers': 2, 'epochs': 5, 'compile_mode': False, 'profile_run': False}

In [46]:
import torch.nn as nn
loss_fn = nn.CrossEntropyLoss()
def train_one_epoch(model, loader, optimizer, device, log_every=200):
    model.train()
    data_time = 0
    compute_time = 0
    epoch_loss = 0.0
    correct = 0
    total = 0
    start_t = time.time()
    data_load_start_time = start_t

    for step, batch in enumerate(loader, start=1):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        if device.type == "cuda":
            torch.cuda.synchronize()
        data_time += time.time() - data_load_start_time

        optimizer.zero_grad(set_to_none=True)

        compute_start_time = time.time()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss # average loss in batch
        logits = outputs.logits

        loss.backward()
        optimizer.step()
        if device.type == "cuda":
            torch.cuda.synchronize()
        compute_time += time.time() - compute_start_time

        batch_size = labels.size(0)
        epoch_loss += loss.item() * batch_size # add sum of losses in batch to total
        c, n = accuracy_from_logits(logits, labels)
        correct += c
        total += n

        if log_every and (step % log_every == 0):
            elapsed = time.time() - start_t
            print(
                f"  step {step:5d}/{len(loader)} "
                f"loss {(epoch_loss/total):.4f} acc {(correct/total):.4f} "
                f"time {elapsed:.1f}s"
            )
        data_load_start_time = time.time()

    return (epoch_loss / total), (correct / total), data_time, compute_time

In [47]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        logits = outputs.logits
        c, n = accuracy_from_logits(logits, labels)
        correct += c
        total += n

    return (correct / total)

In [ ]:
# for epoch in range(1, wandb.config.epochs + 1):
for epoch in range(1, 6):
    epoch_time_start = time.time()

    train_loss, train_acc, data_time, compute_time = train_one_epoch(
        model, train_loader, optimizer, device, log_every=50
    )
    test_acc = evaluate(model, test_loader, device)

    epoch_time = time.time() - epoch_time_start

    print(
        f"Epoch {epoch:02d} | "
        f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
        f"acc {test_acc:.4f} | "
        f"time {epoch_time:.1f}s"
    )

    wandb.log({
        "train/loss": train_loss,
        "train/acc": train_acc,
        "test/acc": test_acc,
        "time/data_loading": data_time,
        "time/compute": compute_time,
        "time/epoch": epoch_time,
    }, step=epoch)

  step    50/782 loss 0.6982 acc 0.4850 time 33.5s
  step   100/782 loss 0.6969 acc 0.4984 time 69.0s
  step   150/782 loss 0.6980 acc 0.4906 time 107.1s
  step   200/782 loss 0.6981 acc 0.4902 time 145.8s
  step   250/782 loss 0.6973 acc 0.4953 time 184.0s
  step   300/782 loss 0.6974 acc 0.4947 time 222.6s
  step   350/782 loss 0.6971 acc 0.4960 time 260.8s
  step   400/782 loss 0.6970 acc 0.4956 time 299.2s
  step   450/782 loss 0.6969 acc 0.4951 time 337.7s
  step   500/782 loss 0.6967 acc 0.4964 time 376.2s
  step   550/782 loss 0.6964 acc 0.4982 time 414.6s
  step   600/782 loss 0.6962 acc 0.5003 time 453.1s
  step   650/782 loss 0.6961 acc 0.5009 time 491.6s
  step   700/782 loss 0.6962 acc 0.5004 time 530.0s
  step   750/782 loss 0.6962 acc 0.5006 time 568.5s
Epoch 01 | train loss 0.6962 acc 0.5005 | acc 0.5000 | time 806.9s
  step    50/782 loss 0.6966 acc 0.4938 time 38.5s
  step   100/782 loss 0.6962 acc 0.4963 time 77.0s
  step   150/782 loss 0.6965 acc 0.4977 time 115.4s
 

In [ ]:
wandb.finish()